# `klygo.files`: Metadata

Inspect file attributes, byte sizes, checksums, and equality without changing source files.

Run this notebook from top to bottom. It creates only temporary example files.

In [ ]:
!pip install git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare sample files

Run this setup again to reset the examples. No Drive mount or manual upload is required.

In [ ]:
from tempfile import TemporaryDirectory
from klygo import files

# Run this cell again to replace only this notebook's temporary workspace.
if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-files-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

In [ ]:
sample_file = root / "sample.bin"
sample_file.write_bytes(b"model weights example\n")
duplicate = root / "duplicate.bin"
duplicate.write_bytes(sample_file.read_bytes())
different = root / "different.bin"
different.write_bytes(b"different bytes")
print("Sample files ready:", root)

## Jump to an API

[`info`](#info) · [`size`](#size) · [`hash`](#hash) · [`compare`](#compare)

<a id="info"></a>

### `files.info`

Collect common metadata for a file or directory.

**Signature:** `files.info(path: 'PathInput', *, verbose: 'bool' = False) -> 'Dict[str, Any]'`

- `path`: Path to inspect or operate on.
- `verbose`: Print the metadata as a Rich table while returning the dictionary.

**Returns:** `dict[str, Any]` — Filesystem metadata with stable keys.

The dictionary has `name`, `stem`, `extension`, `parent`, `size`, `human_size`, `is_file`, `is_dir`, `created`, and `modified`. `parent` is a `Path`; timestamps are local `datetime` values. `verbose=True` prints a Rich table and still returns the dictionary. No checksum is computed.

In [ ]:
metadata = files.info(sample_file, verbose=True)
print(metadata['name'], metadata['size'], metadata['is_file'])
print(sorted(metadata))

**Another useful pattern**

In [ ]:
directory_info = files.info(root)
print('Directory:', directory_info['name'])
print('Bytes below directory:', directory_info['size'])
print('Modified at:', directory_info['modified'])

**Expected error and recovery**

In [ ]:
try:
    files.info(root / "missing.bin")
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

<a id="size"></a>

### `files.size`

Calculate the size of a file or directory tree.

**Signature:** `files.size(path: 'PathInput', human: 'bool' = False) -> 'Union[int, str]'`

- `path`: Path to inspect or operate on.
- `human`: Return a human-readable size rather than an integer.

**Returns:** `int or str` — Bytes, or a formatted size when ``human=True``.

Directory size is the sum of descendant regular-file byte sizes. `human=True` returns a formatted string; use the default integer for arithmetic.

In [ ]:
print(files.size(sample_file), 'bytes')
print(files.size(root, human=True))

**Another useful pattern**

In [ ]:
for item in (sample_file, root):
    print(item.name, files.size(item), 'bytes /', files.size(item, human=True))

**Expected error and recovery**

In [ ]:
try:
    files.size(root / "missing.bin")
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

<a id="hash"></a>

### `files.hash`

Calculate a streaming checksum for a file.

**Signature:** `files.hash(path: 'PathInput', algorithm: 'str' = 'sha256') -> 'str'`

- `path`: Path to inspect or operate on.
- `algorithm`: Algorithm accepted by ``hashlib.new``.

**Returns:** `str` — Lowercase hexadecimal checksum.

The file is read in chunks. `algorithm` is passed to `hashlib.new`; unknown names raise `ValueError`. Hashing a directory also raises `ValueError`.

In [ ]:
digest = files.hash(sample_file)
print('SHA-256:', digest)
print('SHA-1:', files.hash(sample_file, algorithm='sha1'))

**Another useful pattern**

In [ ]:
print('Matching copies:', files.hash(sample_file) == files.hash(duplicate))
print('Changed file:', files.hash(different))

**Expected error and recovery**

In [ ]:
try:
    files.hash(root)
except ValueError as error:
    print("ValueError:", error)

try:
    files.hash(sample_file, algorithm="not-a-hash")
except ValueError as error:
    print("ValueError:", error)

<a id="compare"></a>

### `files.compare`

Compare two files by checksum or binary content.

**Signature:** `files.compare(path1: 'PathInput', path2: 'PathInput', by: 'str' = 'content') -> 'bool'`

- `path1`: Path to the first file.
- `path2`: Path to the second file.
- `by`: File comparison strategy.

**Returns:** `bool` — Whether both files contain identical bytes.

The default `by='content'` compares byte streams after a size check. `by='hash'` compares SHA-256 digests. Both inputs should be files.

In [ ]:
print(files.compare(sample_file, duplicate, by='content'))
print(files.compare(sample_file, different, by='hash'))

**Another useful pattern**

In [ ]:
for mode in ('content', 'hash'):
    print(mode, files.compare(sample_file, duplicate, by=mode))
print('Different:', files.compare(sample_file, different))

**Expected error and recovery**

In [ ]:
try:
    files.compare(sample_file, duplicate, by="name")
except ValueError as error:
    print("ValueError:", error)

## Put it together

In [ ]:
if files.compare(sample_file, duplicate):
    print('Duplicate confirmed; checksum:', files.hash(duplicate))
print('Folder bytes:', files.size(root))

## Remember

`size(..., human=True)` is display text. Use raw byte counts or `compare()` for program logic; use `hash()` when you need a digest.

The examples above use this temporary directory only. See [the overview](overview.ipynb) for the full learning path.